In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2025-12-11 14:07:26.088628: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-11 14:07:26.612354: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-11 14:07:28.167430: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:

conv_base = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(300, 300, 3))

/tmp/ipykernel_3060/4142512147.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  conv_base = keras.applications.MobileNetV2(
I0000 00:00:1765406870.304019    3060 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3536 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [3]:
conv_base.summary()

Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 300,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 150, 150,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 150, 150,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 150, 150,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 150, 150,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 150, 150,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 150, 150,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 150, 150,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 150, 150,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 150, 150,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 150, 150,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 150, 150,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 151, 151,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 75, 75,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 75, 75,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 75, 75,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 75, 75,    │      2,304 │ block_1_depthwis

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 2,223,872 (8.48 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [4]:
conv_base.trainable = True
print("This is the number of trainable weights before freezing the conv base:", len(conv_base.trainable_weights))


This is the number of trainable weights before freezing the conv base: 156


In [5]:
conv_base.trainable = False
print("This is the number of trainable weights after freezing the conv base:", len(conv_base.trainable_weights))

This is the number of trainable weights after freezing the conv base: 0


Data Augmentation using Keras preprocessing layers

In [6]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomBrightness(0.2),
])

Building the Model with MobileNetV2 as the base convolutional layer

In [7]:
inputs = keras.Input(shape=(300, 300, 3))
x = data_augmentation(inputs)

x = keras.applications.mobilenet_v2.preprocess_input(x)
x = conv_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

Loading Dataset using image_dataset_from_directory

In [8]:
import pathlib
import os

# 1. Start with '/mnt/c/' (or '/mnt/d/' if it's on D drive)
# 2. Add the rest of the Windows path using forward slashes
# Example: If on Windows it is "C:\Users\Name\Downloads\Casting..."
path_str = "/mnt/d/Projects/Models/DefectDetectionEdgeLens/CastingProductImageDataset/casting_512x512"

data_dir = pathlib.Path(path_str)

In [9]:
# --- Training Set (80% of data) ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',          # Automatically infer labels from subdirectories
    label_mode='binary',        # Binary classification (defect vs ok)
    image_size=(300, 300),      # Resizing for MobileNetV2 inputs
    batch_size=16,              # Safe size for your 6GB GPU
    color_mode='rgb',           # Autoconvert Grayscale -> RGB (3 channels)
    shuffle=True,
    seed=123
)

Found 1300 files belonging to 2 classes.


In [10]:
# --- Test Set ---
test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels='inferred',
    label_mode='binary',
    image_size=(300, 300),
    batch_size=16,
    color_mode='rgb',
    shuffle=False  # Don't shuffle test data
)

Found 1300 files belonging to 2 classes.


In [11]:
class_names = train_ds.class_names
print(f"Classes found: {class_names}")

Classes found: ['def_front', 'ok_front']


Compiling the model with Adam optimizer and binary crossentropy loss function

In [12]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
windows_path = "/mnt/d/Projects/Models/DefectDetectionEdgeLens/defect_detection_mobilenetv2.h5"

callbacks = [
    # Stop if validation loss doesn't improve for 3 epochs
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    # Save directly to Windows D: drive (example) 
    keras.callbacks.ModelCheckpoint(windows_path, save_best_only=True)
]

In [14]:
history = model.fit(
    train_ds,
    epochs=20,            # It will likely stop earlier due to EarlyStopping
    validation_data=test_ds,
    callbacks=callbacks
)

Epoch 1/20


2025-12-10 22:49:34.238611: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91700


82/82 ━━━━━━━━━━━━━━━━━━━━ 12s 96ms/step - accuracy: 0.7238 - loss: 0.5333 - val_accuracy: 0.7869 - val_loss: 0.4205
Epoch 2/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step - accuracy: 0.8338 - loss: 0.3898 - val_accuracy: 0.8838 - val_loss: 0.3325
Epoch 3/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 85ms/step - accuracy: 0.8700 - loss: 0.3238 - val_accuracy: 0.9031 - val_loss: 0.2877
Epoch 4/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 122ms/step - accuracy: 0.8938 - loss: 0.2944 - val_accuracy: 0.9177 - val_loss: 0.2610
Epoch 5/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 82ms/step - accuracy: 0.8954 - loss: 0.2699 - val_accuracy: 0.9192 - val_loss: 0.2348
Epoch 6/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step - accuracy: 0.9108 - loss: 0.2575 - val_accuracy: 0.9285 - val_loss: 0.2180
Epoch 7/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 82ms/step - accuracy: 0.9323 - loss: 0.2280 - val_accuracy: 0.9338 - val_loss: 0.2167
Epoch 8/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 7s 82ms/step - accuracy: 0.9154 - loss: 0.2398 - val_accuracy: 0.9331 - val_loss: